## Step One : Add dependencies

In [1]:
import matplotlib
import pandas as pd


Making the variable in different df

In [2]:
orders = pd.read_excel('data/data.xlsx', sheet_name='Sales Order_data')
territories = pd.read_excel('data/data.xlsx', sheet_name='Sales Territory_data')
sales = pd.read_excel('data/data.xlsx', sheet_name='Sales_data')
resellers = pd.read_excel('data/data.xlsx', sheet_name='Reseller_data')
dates = pd.read_excel('data/data.xlsx', sheet_name='Date_data')
products = pd.read_excel('data/data.xlsx', sheet_name='Product_data')
customers = pd.read_excel('data/data.xlsx', sheet_name='Customer_data')

In [4]:
dataframes = {
    'orders': orders,
    'territories': territories,
    'sales': sales,
    'resellers': resellers,
    'dates': dates,
    'products': products,
    'customers': customers,
}

for name, df in dataframes.items():
    print(f"\n{'=' * 60}")
    print(f"DataFrame : {name}")
    print(f"{'=' * 60}")
    print("\n--- 5 premières lignes ---")
    print(df.head())
    print("\n--- Types de colonnes ---")
    print(df.dtypes)


DataFrame : orders

--- 5 premières lignes ---
    Channel  SalesOrderLineKey Sales Order Sales Order Line
0  Reseller           43659001     SO43659      SO43659 - 1
1  Reseller           43659002     SO43659      SO43659 - 2
2  Reseller           43659003     SO43659      SO43659 - 3
3  Reseller           43659004     SO43659      SO43659 - 4
4  Reseller           43659005     SO43659      SO43659 - 5

--- Types de colonnes ---
Channel                str
SalesOrderLineKey    int64
Sales Order            str
Sales Order Line       str
dtype: object

DataFrame : territories

--- 5 premières lignes ---
   SalesTerritoryKey     Region        Country          Group
0                  1  Northwest  United States  North America
1                  2  Northeast  United States  North America
2                  3    Central  United States  North America
3                  4  Southwest  United States  North America
4                  5  Southeast  United States  North America

--- Types de colo

In [5]:
for name, df in dataframes.items():
    print(f"\n{'=' * 60}")
    print(f"DataFrame : {name}")
    print(f"{'=' * 60}")

    print("\n--- info() : taille, types, valeurs non nulles ---")
    df.info()

    print("\n--- describe() : statistiques descriptives ---")
    print(df.describe(include='all'))


DataFrame : orders

--- info() : taille, types, valeurs non nulles ---
<class 'pandas.DataFrame'>
RangeIndex: 121253 entries, 0 to 121252
Data columns (total 4 columns):
 #   Column             Non-Null Count   Dtype
---  ------             --------------   -----
 0   Channel            121253 non-null  str  
 1   SalesOrderLineKey  121253 non-null  int64
 2   Sales Order        121253 non-null  str  
 3   Sales Order Line   121253 non-null  str  
dtypes: int64(1), str(3)
memory usage: 3.7 MB

--- describe() : statistiques descriptives ---
         Channel  SalesOrderLineKey Sales Order Sales Order Line
count     121253       1.212530e+05      121253           121253
unique         2                NaN       31455           121253
top     Reseller                NaN     SO51721      SO43659 - 1
freq       60855                NaN          72                1
mean         NaN       5.782642e+07         NaN              NaN
std          NaN       9.009990e+06         NaN              Na

## Colonnes au type potentiellement incorrect

| Table | Colonne | Actuel | À corriger en |
|-------|---------|--------|---------------|
| sales | ShipDateKey | float64 | Int64 (il y a des NaN) |
| sales | Unit Price Discount Pct | int64 | float64 (c'est un %) |
| sales | OrderDateKey, DueDateKey | int64 | datetime si on veut faire des calculs de dates |
| dates | DateKey | int64 | ok pour joindre, sinon datetime via la colonne Date |



In [8]:
# Vérification rapide des colonnes suspectes

# ShipDateKey : float à cause de valeurs manquantes ?
print("sales — ShipDateKey")
print(f"  Type : {sales['ShipDateKey'].dtype}")
print(f"  Valeurs manquantes : {sales['ShipDateKey'].isna().sum()}")
print(sales['ShipDateKey'].head())

print("\nsales — Unit Price Discount Pct")
print(f"  Type : {sales['Unit Price Discount Pct'].dtype}")
print(f"  Valeurs uniques (échantillon) : {sales['Unit Price Discount Pct'].unique()[:10]}")

print("\nsales — Clés de date (format entier YYYYMMDD)")
print(sales[['OrderDateKey', 'DueDateKey', 'ShipDateKey']].dtypes)

sales — ShipDateKey
  Type : float64
  Valeurs manquantes : 2113
0    20170709.0
1    20170709.0
2    20170709.0
3    20170709.0
4    20170709.0
Name: ShipDateKey, dtype: float64

sales — Unit Price Discount Pct
  Type : int64
  Valeurs uniques (échantillon) : [0]

sales — Clés de date (format entier YYYYMMDD)
OrderDateKey      int64
DueDateKey        int64
ShipDateKey     float64
dtype: object


In [ ]:
cost = products['Standard Cost']

print("=== Coût standard — tous les produits ===")
print(f"Médiane    : {cost.median():.2f}")
print(f"Écart-type : {cost.std():.2f}")
print(f"Variance   : {cost.var():.2f}")

print("\n=== Coût standard — par Category ===")
cost_by_category = products.groupby('Category')['Standard Cost'].agg(
    median='median',
    ecart_type='std',
    variance='var'
)
print(cost_by_category.round(2))

=== Coût standard — tous les produits ===
Médiane    : 204.63
Écart-type : 497.08
Variance   : 247085.43

=== Coût standard — par Category ===
             median  ecart_type   variance
Category                                  
Accessories   11.22       13.13     172.34
Bikes        713.08      552.41  305153.73
Clothing      26.18       12.42     154.37
Components   187.16      280.64   78756.88


In [ ]:

FILL_VALUES = {
    'products': {'Color': 'N/A'},
    'sales': {},  # ShipDateKey traité juste après avec OrderDateKey
}

for name, df in dataframes.items():
    missing = {}
    for col in df.columns:
        n_nan = df[col].isna().sum()
        n_empty = (df[col] == '').sum() if df[col].dtype == object or str(df[col].dtype) == 'str' else 0
        if n_nan or n_empty:
            missing[col] = {'NaN': int(n_nan), 'vide': int(n_empty)}

    if missing:
        print(f"\n{name} — colonnes avec valeurs vides :")
        for col, counts in missing.items():
            print(f"  [{col}]  NaN={counts['NaN']}, vide={counts['vide']}")

        for col, fill in FILL_VALUES.get(name, {}).items():
            if col in df.columns:
                df[col] = df[col].fillna(fill).replace('', fill)

    else:
        print(f"\n{name} : aucune cellule vide")

# ShipDateKey manquant → on reprend la date de commande
sales['ShipDateKey'] = sales['ShipDateKey'].fillna(sales['OrderDateKey'])

print("\n--- Après remplacement ---")
for name in ['products', 'sales']:
    df = dataframes[name]
    remaining = df.isna().sum()
    remaining = remaining[remaining > 0]
    print(f"{name} : {dict(remaining) if len(remaining) else 'ok'}")


orders : aucune cellule vide

territories : aucune cellule vide

sales — colonnes avec valeurs vides :
  [ShipDateKey]  NaN=2113, vide=0

resellers : aucune cellule vide

dates : aucune cellule vide

products — colonnes avec valeurs vides :
  [Color]  NaN=56, vide=0

customers : aucune cellule vide

--- Après remplacement ---
products : ok
sales : ok


In [11]:
# 2. Vérification du total des ventes (remise incluse)

discount = sales['Unit Price Discount Pct'].astype(float)

# Extended Amount = prix unitaire × quantité
sales['expected_extended'] = sales['Unit Price'] * sales['Order Quantity']
ext_ok = (sales['Extended Amount'] - sales['expected_extended']).abs() < 0.01

# Sales Amount = montant étendu × (1 - remise)
sales['expected_sales'] = sales['Extended Amount'] * (1 - discount)
sales_ok = (sales['Sales Amount'] - sales['expected_sales']).abs() < 0.01

print(f"Lignes avec remise > 0 : {(discount > 0).sum()}")
print(f"Extended Amount incorrect : {(~ext_ok).sum()} lignes")
print(f"Sales Amount incorrect      : {(~sales_ok).sum()} lignes")

if (~sales_ok).any():
    print("\nExemples d'écarts (remise non nulle en priorité) :")
    bad = sales[~sales_ok].copy()
    bad = bad.sort_values('Unit Price Discount Pct', ascending=False)
    print(bad[['Unit Price', 'Order Quantity', 'Extended Amount', 'Unit Price Discount Pct',
               'Sales Amount', 'expected_sales']].head())

sales.drop(columns=['expected_extended', 'expected_sales'], inplace=True)

Lignes avec remise > 0 : 0
Extended Amount incorrect : 0 lignes
Sales Amount incorrect      : 3282 lignes

Exemples d'écarts (remise non nulle en priorité) :
     Unit Price  Order Quantity  Extended Amount  Unit Price Discount Pct  \
656   1971.9942              12       23663.9304                        0   
658   1957.4942              13       25447.4246                        0   
713      5.0136              13          65.1768                        0   
734      5.2250              21         109.7250                        0   
769   1971.9942              14       27607.9188                        0   

     Sales Amount  expected_sales  
656    23190.6518      23663.9304  
658    24938.4761      25447.4246  
713       63.8733         65.1768  
734      104.2388        109.7250  
769    27055.7604      27607.9188  


In [ ]:
# 3. Nouvelles colonnes + correction des types

# Price ratio : prix affiché / coût standard (catalogue produits)
products['Price ratio'] = products['List Price'] / products['Standard Cost']

# Line number : numéro après le tiret dans "SO43659 - 1"
orders['Line number'] = orders['Sales Order Line'].str.split(' - ').str[-1].astype(int)

# Margin : ventes − coût total
sales['Margin'] = sales['Sales Amount'] - sales['Total Product Cost']

# Types à corriger pour l'analyse
sales['Unit Price Discount Pct'] = sales['Unit Price Discount Pct'].astype('float64')
sales['ShipDateKey'] = sales['ShipDateKey'].astype('Int64')
orders['Channel'] = orders['Channel'].astype('category')

print("products['Price ratio'].describe()")
print(products['Price ratio'].describe().round(2))
print("\norders['Line number'] — aperçu")
print(orders[['Sales Order Line', 'Line number']].head())
print("\nsales['Margin'].describe()")
print(sales['Margin'].describe().round(2))
print("\nTypes corrigés (sales) :")
print(sales[['ShipDateKey', 'Unit Price Discount Pct', 'Margin']].dtypes)

products['Price ratio'].describe()
count    397.00
mean       1.93
std        0.37
min        1.30
25%        1.65
50%        1.80
75%        2.25
max        2.80
Name: Price ratio, dtype: float64

orders['Line number'] — aperçu
  Sales Order Line  Line number
0      SO43659 - 1            1
1      SO43659 - 2            2
2      SO43659 - 3            3
3      SO43659 - 4            4
4      SO43659 - 5            5

sales['Margin'].describe()
count    121253.00
mean        103.51
std         399.12
min      -15099.75
25%           2.50
50%          15.33
75%          61.55
max        1487.84
Name: Margin, dtype: float64

Types corrigés (sales) :
ShipDateKey                  Int64
Unit Price Discount Pct    float64
Margin                     float64
dtype: object
